In [5]:
import requests
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [6]:
df = pd.read_excel("Реестр (1).xlsx", header = 2)

**Вырезаем столбик с ИНН**

In [7]:
correct_df = df[df['Основной вид деятельности'].str.contains('41.2', na=False)]
correct_df = correct_df[correct_df['Тип субъекта'] == 'Юридическое лицо']
correct_df = correct_df[(correct_df['Категория'] == 'Малое предприятие') | (correct_df['Категория'] == 'Среднее предприятие')]


In [8]:
INN = correct_df['ИНН'].reset_index(drop = True)

In [9]:
INN

0        5038038838
1        5027064258
2        5027006369
3        7701651356
4        1414006922
            ...    
10568    6504043928
10569    2312105041
10570    2301032458
10571    2002001476
10572     702005585
Name: ИНН, Length: 10573, dtype: int64

**Парсим JSON с инфой и достаем чистую прибыль по годам**

In [14]:
session = requests.Session()

#Правильные заголовки
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15",
    "Referer": "https://bo.nalog.gov.ru/",
})

#Куки
session.get("https://bo.nalog.gov.ru/", timeout=10)

years = ['2021', '2022', '2023', '2024', '2025']
dfs = {year: pd.DataFrame() for year in years}

#Получаем id
for i in range(len(INN)):
    try:
        inn = INN[i]
        res_id = session.get(
            f"https://bo.nalog.gov.ru/advanced-search/organizations/search?query={inn}&page=0&size=20",
            timeout=10
        ).json()
        if not res_id.get("content"):
            continue

        org_id = res_id["content"][0]["id"]

        #Получаем всю инфу в JSON
        res = session.get(
            f"https://bo.nalog.gov.ru/nbo/organizations/{org_id}/bfo/",
            timeout=10
        ).json()

        if not res:
            continue



        data_by_year = {}
        for item in res:
            try:
                year = item['period']

                fr = item['typeCorrections'][0]['correction']['financialResult']
                address = item['organizationInfo']['address']
                company_name = item['organizationInfo']['fullName']

                data_by_year[year] = fr.copy()
                data_by_year[year]['address'] = address
                data_by_year[year]['company_name'] = company_name
            except (KeyError, IndexError, TypeError):
                continue



        for year in years:
            if year in data_by_year:
                for key, value in data_by_year[year].items():
                    dfs[year].loc[i, key] = value
        
    except (KeyError, IndexError, TypeError):
        pass

    if (i%100 == 0):
        print(i)


for year in years:
    dfs[year].to_csv(f'financial_data_{year}.csv', index = False, encoding = 'utf-8')
print('All done, master')

0
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900
2000
2100
2200
2300
2400
2500
2600
2700
2800
2900
3000
3100
3200
3300
3400
3500
3600
3700
3800
3900
4000
4100
4200
4300
4400
4500
4600
4700
4800
4900
5000
5100
5200
5300
5400
5500
5600
5700
5800
5900
6000
6100
6200
6300
6400
6500
6600
6700
6800
6900
7000
7100
7200
7300
7400
7500
7600
7700
7800
7900
8000
8100
8200
8300
8400
8500
8600
8700
8800
8900
9000
9100
9200
9300
9400
9500
9600
9700
9800
9900
10000
10100
10200
10300
10400
10500
All done, master
